# 🔢 Chapter 8 — Data Analysis Project: Pure NumPy

---

### 🎯 Complete Project (No Pandas!)

We'll generate a synthetic employee dataset and perform full analysis using **only NumPy**:
1. Generate and save data
2. Descriptive statistics
3. Correlation analysis
4. Department-wise aggregations
5. Outlier detection (z-scores)
6. Binning continuous data
7. Top/Bottom performers

## Step 1: Generate Synthetic Employee Dataset

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 1000

# Generate data
ages = rng.integers(22, 65, n)
departments = rng.choice([1, 2, 3, 4, 5], n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
years_exp = np.clip(ages - 22 + rng.integers(-2, 5, n), 0, 40)
base_salary = 30000 + years_exp * 2000 + rng.normal(0, 5000, n)
salary = np.clip(base_salary, 25000, 200000).astype(int)
performance = np.clip(rng.normal(3.0, 0.8, n) + years_exp * 0.02, 1.0, 5.0)
performance = np.round(performance, 2)

# Stack into a 2D array
# Columns: age, department, years_exp, salary, performance
data = np.column_stack([ages, departments, years_exp, salary, performance])
col_names = ["Age", "Department", "YearsExp", "Salary", "Performance"]

# Save and load
np.savetxt("employees.csv", data, delimiter=",",
           header=",".join(col_names), comments="", fmt=["%.0f","%.0f","%.0f","%.0f","%.2f"])

# Also save in NumPy format
np.save("employees.npy", data)

print(f"Dataset shape: {data.shape}")
print(f"Columns: {col_names}")
print(f"\nFirst 5 rows:")
for i in range(5):
    print(f"  {data[i]}")

## Step 2: Descriptive Statistics

In [ ]:
# Load data
data = np.load("employees.npy")
col_names = ["Age", "Department", "YearsExp", "Salary", "Performance"]

print(f"{'Metric':<12}", end="")
for name in col_names:
    print(f"{name:>12}", end="")
print()
print("-" * 72)

for label, func in [("Mean", np.mean), ("Std", np.std),
                     ("Min", np.min), ("Max", np.max),
                     ("Median", np.median)]:
    print(f"{label:<12}", end="")
    for j in range(data.shape[1]):
        val = func(data[:, j])
        print(f"{val:>12.2f}", end="")
    print()

# Percentiles
for pct in [25, 50, 75]:
    print(f"P{pct:<10}", end="")
    for j in range(data.shape[1]):
        val = np.percentile(data[:, j], pct)
        print(f"{val:>12.2f}", end="")
    print()

## Step 3: Correlation Matrix

In [ ]:
corr = np.corrcoef(data, rowvar=False)

print(f"{'':>12}", end="")
for name in col_names:
    print(f"{name:>12}", end="")
print()
print("-" * 72)

for i, name in enumerate(col_names):
    print(f"{name:>12}", end="")
    for j in range(len(col_names)):
        print(f"{corr[i, j]:>12.3f}", end="")
    print()

# Key findings
age_sal = corr[0, 3]
exp_sal = corr[2, 3]
exp_perf = corr[2, 4]
print(f"\n📊 Key correlations:")
print(f"  Age ↔ Salary:       {age_sal:+.3f}")
print(f"  Experience ↔ Salary: {exp_sal:+.3f}")
print(f"  Experience ↔ Perf:   {exp_perf:+.3f}")

## Step 4: Department-Wise Analysis

In [ ]:
dept_names = {1: "Engineering", 2: "Marketing", 3: "Sales", 4: "HR", 5: "Finance"}
dept_col = data[:, 1]
salary_col = data[:, 3]
perf_col = data[:, 4]

print(f"{'Department':<15} {'Count':>6} {'Avg Salary':>12} {'Avg Perf':>10} {'Max Salary':>12}")
print("-" * 58)

for dept_id in sorted(np.unique(dept_col)):
    mask = dept_col == dept_id
    count = mask.sum()
    avg_salary = salary_col[mask].mean()
    avg_perf = perf_col[mask].mean()
    max_salary = salary_col[mask].max()
    print(f"{dept_names[dept_id]:<15} {count:>6} ${avg_salary:>11,.0f} {avg_perf:>10.2f} ${max_salary:>11,}")

## Step 5: Outlier Detection (Z-Scores)

In [ ]:
salary_col = data[:, 3]

# Z-score = (x - mean) / std
z_scores = (salary_col - salary_col.mean()) / salary_col.std()

# Outliers: |z| > 2
outlier_mask = np.abs(z_scores) > 2
n_outliers = outlier_mask.sum()

print(f"Salary statistics:")
print(f"  Mean: ${salary_col.mean():,.0f}")
print(f"  Std:  ${salary_col.std():,.0f}")
print(f"\nOutliers (|z| > 2): {n_outliers} employees ({n_outliers/len(salary_col)*100:.1f}%)")

# Show outlier details
if n_outliers > 0:
    outlier_data = data[outlier_mask]
    print(f"\nOutlier salaries: {sorted(outlier_data[:, 3].astype(int))[:10]}...")

## Step 6: Binning Continuous Data

In [ ]:
age_col = data[:, 0]

# Create bins
bins = [20, 30, 40, 50, 65]
bin_labels = ["20-29", "30-39", "40-49", "50-64"]
bin_indices = np.digitize(age_col, bins) - 1  # 0-indexed

print(f"{'Age Group':<12} {'Count':>6} {'Avg Salary':>12} {'Avg Perf':>10}")
print("-" * 44)

for i, label in enumerate(bin_labels):
    mask = bin_indices == i
    if mask.any():
        count = mask.sum()
        avg_sal = data[mask, 3].mean()
        avg_perf = data[mask, 4].mean()
        print(f"{label:<12} {count:>6} ${avg_sal:>11,.0f} {avg_perf:>10.2f}")

## Step 7: Top/Bottom Performers

In [ ]:
perf_col = data[:, 4]
salary_col = data[:, 3]

# Top 10 performers
top_indices = np.argsort(perf_col)[-10:][::-1]
print("🏆 Top 10 Performers:")
print(f"{'#':>3} {'Age':>5} {'Dept':>12} {'Exp':>5} {'Salary':>10} {'Perf':>6}")
for rank, idx in enumerate(top_indices, 1):
    row = data[idx]
    dept = {1:"Engineering",2:"Marketing",3:"Sales",4:"HR",5:"Finance"}[row[1]]
    print(f"{rank:>3} {row[0]:>5.0f} {dept:>12} {row[2]:>5.0f} ${row[3]:>9,.0f} {row[4]:>6.2f}")

# Bottom 5
bottom_indices = np.argsort(perf_col)[:5]
print("\n⚠️ Bottom 5 Performers:")
for idx in bottom_indices:
    row = data[idx]
    dept = {1:"Engineering",2:"Marketing",3:"Sales",4:"HR",5:"Finance"}[row[1]]
    print(f"  Age={row[0]:.0f}, Dept={dept}, Exp={row[2]:.0f}yrs, "
          f"Salary=${row[3]:,.0f}, Perf={row[4]:.2f}")

## 🎓 Summary

In this chapter we performed **complete data analysis using only NumPy** — no Pandas needed!

### Techniques Used:
- ✅ Data generation with `np.random.default_rng`
- ✅ File I/O: `np.savetxt`, `np.save`, `np.load`
- ✅ Descriptive statistics: mean, std, percentiles
- ✅ Correlation matrix with `np.corrcoef`
- ✅ Group-by aggregations using boolean masking + `np.unique`
- ✅ Outlier detection with z-scores
- ✅ Binning with `np.digitize`
- ✅ Sorting & ranking with `np.argsort`

In [ ]:
# Cleanup
import os
for f in ["employees.csv", "employees.npy"]:
    if os.path.exists(f):
        # os.remove(f)  # Uncomment to clean up
        pass
print("Generated files: employees.csv, employees.npy")
